# Surface-enumeration Python API

This tutorial prepares a vacancy enumeration for a lithium (100) slab using the public Python API. 

In [ ]:
from pathlib import Path
from shutil import which

from pymatgen.core import Structure

from surface_pd.core import EnumerationSlab, EnumWithComposition

enumlib_available = which("enum.x") is not None and any(
    which(name) is not None
    for name in ("makestr.x", "makeStr.x", "makeStr.py")
)

The notebook expects to be started from the `examples/` directory. 

## 1. Load the slab structure

The VASP file `POSCAR_LI_100.vasp` contains a nine-layer Li(100) slab, vacuum normal to direction 2 (the third lattice vector), and selective-dynamics flags that identify its fixed central region and relaxed surfaces. The surface-pd slab enumeration builds on functionality in the pymatgen package. Hence, we first load the slab structure using pymatgen.

In [ ]:
structure_path = Path(
    "enumeration-examples/structure/electrode/POSCAR_Li_100.vasp"
)
structure = Structure.from_file(structure_path)

print("formula:", structure.composition.reduced_formula)
print("sites:", len(structure))
print("selective dynamics:", "selective_dynamics" in structure.site_properties)
structure

## 2. Configure an `EnumerationSlab`

`EnumerationSlab.from_structure()` preserves the pymatgen structure data while adding surface-enumeration settings. Here `direction=2` selects the third lattice direction as the surface normal, `enumerated_species` selects Li, and `num_enumerated_layers={"Li": 1}` selects the outermost Li layer. Layer counts are independent for each selected species: for example, `{"Li": 1, "O": 2}` would select the outermost Li layer and two outermost O layers. With `symmetric=False`, only the top surface is enumerated. Setting `symmetric=True` instead selects corresponding layers on both surfaces and requires a compatible symmetric slab.

In [ ]:
slab = EnumerationSlab.from_structure(
    structure,
    direction=2,
    tolerance=0.03,
    enumerated_species=["Li"],
    num_enumerated_layers={"Li": 1},
    symmetric=False,
)

print("surface normal direction:", slab.direction)
print("selected species:", slab.enumerated_species)
print("selected layers:", slab.num_enumerated_layers)
print("symmetric enumeration:", slab.symmetric)

## 3. Inspect layers and selected sites

Layer identification groups sites by fractional height within `tolerance`. `get_enumerated_site_indices()` reports exactly which original site indices may change, grouped by species. `get_fixed_region_bounds()` separately reports the central fixed region inferred from selective-dynamics flags. Site selection does not modify the slab.

In [ ]:
layers = slab.layers_finder()
selected_indices = slab.get_enumerated_site_indices()
center_bottom, center_top = slab.get_fixed_region_bounds()

print("Li layer heights:", list(layers["Li"]))
print("fixed-region bounds:", (center_bottom, center_top))
print("selected site indices:", selected_indices)

## 4. Define the ordered surface configurations

The replacement mapping below requests 50% Li occupancy on the selected surface Li sublattice; the missing 50% represents vacancies. A mapping such as `{"Li": {"Li": 0.5, "Na": 0.5}}` would instead enumerate Li/Na substitutions. For each requested composition, enumlib finds symmetrically distinct orderings over the allowed surface cells. This fraction needs an area multiplier of two, so `min_cell_size=max_cell_size=2`. Surface-pd rejects three-dimensional derivatives that multiply or mix the surface-normal vector. Larger in-plane cell multipliers permit more orderings and can greatly increase cost.

In [ ]:
enumerator = EnumWithComposition(
    replacements={"Li": {"Li": 0.5, "Na": 0.5}},
    min_cell_size=2,
    max_cell_size=4,
)

## 5. Run the external enumeration when requested

`EnumWithComposition.apply_enumeration` delegates candidate generation to enumlib, then returns only finalized `EnumerationSlab` objects whose lattice transformations are strictly in plane. Internal marker species used to isolate the selected surface sites are never exposed. The deterministic setup above is always executed by the test suite. External execution is opt-in because enumlib is separately installed. To run it, install `enum.x` and `makestr.x` (or a supported `makeStr` variant), make them discoverable on `PATH`, set `RUN_ENUMLIB = True`, and rerun this cell.

In [ ]:
RUN_ENUMLIB = False

if RUN_ENUMLIB and enumlib_available:
    enumerated_slabs = enumerator.apply_enumeration(
        slab, max_structures=20
    )
    print(f"enumlib returned {len(enumerated_slabs)} surface slab(s)")
    for index, result in enumerate(enumerated_slabs):
        print(index, result.formula, result.lattice.abc)
elif RUN_ENUMLIB:
    print("enumlib was requested but its executables were not found")
else:
    print("enumlib execution skipped by default; deterministic setup complete")

In [ ]:
if RUN_ENUMLIB and enumlib_available:
    top_layer_coord = list(layers["Li"])[0]
    for site in enumerated_slabs[0].sites:
        if abs(site.frac_coords[slab.direction] - top_layer_coord) <= 5e-3:
            print(site.specie, site.coords)

## 6. Symmetric enumeration on both surfaces

Symmetric enumeration requires more than an inversion-symmetric lattice: the selective-dynamics flags must identify relaxed regions on both surfaces around a fixed central region. The one-sided Li(100) slab above is therefore intentionally asymmetric. The committed Li2O(110) slab provides a compatible symmetric example. Its outermost Li layer and two outermost O layers are selected independently on both surfaces.

In [ ]:
symmetric_structure_path = Path(
    "enumeration-examples/structure/electrode/POSCAR_Li2O_110.vasp"
)
symmetric_structure = Structure.from_file(symmetric_structure_path)
symmetric_slab = EnumerationSlab.from_structure(
    symmetric_structure,
    direction=2,
    tolerance=0.03,
    enumerated_species=["Li", "O"],
    num_enumerated_layers={"Li": 1, "O": 2},
    symmetric=True,
)

print(
    "selected symmetric sites:",
    symmetric_slab.get_enumerated_site_indices(),
)

The Li sites remain fully occupied while 75% of the selected O sites remain occupied. Surface-pd first enumerates one surface, restores complete selective-dynamics metadata, mirrors the ordering onto the second surface, and returns only finalized inversion-symmetric slabs.

In [ ]:
symmetric_enumerator = EnumWithComposition(
    replacements={"Li": {"Li": 1.0}, "O": {"O": 0.75}},
    min_cell_size=1,
    max_cell_size=2,
)

In [ ]:
RUN_SYMMETRIC_ENUMLIB = False

if RUN_SYMMETRIC_ENUMLIB and enumlib_available:
    symmetric_results = symmetric_enumerator.apply_enumeration(
        symmetric_slab, max_structures=20
    )
    print(f"enumlib returned {len(symmetric_results)} symmetric slab(s)")
    for index, result in enumerate(symmetric_results):
        print(index, result.formula, result.lattice.abc, result.is_symmetry())
elif RUN_SYMMETRIC_ENUMLIB:
    print(
        "symmetric enumlib execution requested but executables were not found"
    )
else:
    print("symmetric enumlib execution skipped by default")